### Pacific Atlantic Water Flow



Input & Output
Input
heights — an m x n integer matrix representing the height above sea level of each cell


Output
a list of grid coordinates where rain water can flow to both oceans

Example
Input

heights = [[1,2,2,3,5],[3,2,3,4,4],[2,4,5,3,1],[6,7,1,4,5],[5,1,1,2,4]]

Output

[[0,4],[1,3],[1,4],[2,2],[3,0],[3,1],[4,0]]

Example

Input

heights = [[2,1],[1,2]]

Output

[[0,0],[0,1],[1,0],[1,1]]

In [1]:
from typing import List

### Brute Force Approach

In [3]:
from collections import deque
def pacific_atlantic_water_flow(heights: list[list[int]]) -> list[list[int]]:
    # WRITE YOUR BRILLIANT CODE HERE

    # we define a get neigbors function
    # this function will allow us to understand where the rainwater can flow
    # if we are at a cell, the rainwater will go to the neigbor provided it has lesser/equal value to current cell
    def get_neighbors(heights, r, c):
        rows = len(heights)
        cols = len(heights[0])

        neighbors = []
        directions = [
            (-1,0),
            (0, 1),
            (1, 0),
            (0,-1)
        ]

        for dr, dc in directions:
            nr = r + dr
            nc = c + dc
            if 0<=nr<rows and 0<=nc<cols:
                # check if the value of the neighbor is less than or equal to current cell
                if heights[nr][nc] <= heights[r][c]:
                    neighbors.append((nr,nc))

        return neighbors

    # start off by writing a normal bfs function
    # this bfs will return true if the cell we are at is able to drain the rainwater to the destination row or column
    def bfs(heights, start_r, start_c, destination_r, destination_c):
        queue = deque([(start_r, start_c)])
        visited = {(start_r, start_c)}
        
        flag = False
        # reached_desitination = []
        while queue:
            n = len(queue)
            for _ in range(n):
                r,c = queue.popleft()
                if r == destination_r or c == destination_c:
                    flag = True
                    return flag # this will return true, if our current node is accessible to the coast

                for r,c in get_neighbors(heights, r,c):
                    if (r,c) not in visited:
                        queue.append((r,c))
                        visited.add((r,c))
        # if we never encounter true flag, our flag will be set to false and returned as is
        return flag
       
    both_oceans = []
    number_of_rows = len(heights)
    number_of_cols = len(heights)[0]
    # we call the bfs function for atlantic and for pacific on each cell
    for i in range(number_of_rows):
        for j in range(number_of_cols):
            reaches_pacific = bfs(heights, i,j,0,0)
            reaches_atlantic = bfs(heights,i,j,number_of_rows-1,number_of_cols-1)
            if reaches_atlantic and reaches_pacific:
                both_oceans.append([i,j])



    return both_oceans


### Altenate Approach

In [ ]:
# start applying bfs from the atlantic and the pacific shore
# we include neighbors with greater than or equal to heights, all the nodes that can be visited will be in visted set
# whichever nodes are in both the visited sets is our answer

In [ ]:
from collections import deque
def pacific_atlantic_water_flow(heights: list[list[int]]) -> list[list[int]]:
    # WRITE YOUR BRILLIANT CODE HERE

    # defining a get neighbor function
    def get_neighbors(heights, r, c):
        rows = len(heights)
        cols = len(heights[0])

        directions = [
            (-1,0),
            (0, 1),
            (1, 0),
            (0,-1)
        ]
        neighbors = []
        for dr, dc in directions:
            nr = r + dr
            nc = c + dc
            # check indices
            if 0<=nr<rows and 0<=nc<cols:
                # check if the neighbor has greater or equal value
                if heights[nr][nc] >= heights[r][c]:
                    neighbors.append((nr,nc))

        return neighbors

    def bfs(heights, r, c):
        queue = deque([(r,c)])
        visited = {(r,c)}

        while queue:
            n = len(queue)
            for _ in range(n):
                r,c = queue.popleft()
                for nr, nc in get_neighbors(heights, r, c):
                    if (nr,nc) not in visited:
                        queue.append((nr,nc))
                        visited.add((nr,nc))
        # by the end of the function we will get all the nodes that can be reached in this visited set
        return visited
    reaching_pacific = set()
    # rainwater travelling uphill from pacific
    for i in range(len(heights)):
        for j in range(len(heights[0])):
            if i == 0 or j == 0:
                visited_by_bfs = bfs(heights, i, j)
                reaching_pacific.update(visited_by_bfs)

    reaching_atlantic = set()
    # rainwater travelling from atlantic
    for i in range(len(heights)-1, -1, -1):
        for j in range(len(heights[0])-1, -1, -1):
            if i == len(heights)-1 or j == len(heights[0])-1:
                visited_by_bfs = bfs(heights, i, j)
                reaching_atlantic.update(visited_by_bfs)

    output = [list(item) for item in reaching_atlantic.intersection(reaching_pacific)]
    return output

### Optimized Version

In [ ]:
# we can start bfs simultaneously from both the oceans and get their visited sets
from collections import deque
def pacific_atlantic_water_flow(heights: list[list[int]]) -> list[list[int]]:
    def get_neighbors(heights, r, c):
        rows = len(heights)
        cols = len(heights[0])

        directions = [
            (-1,0),
            (0, 1),
            (1, 0),
            (0,-1)
        ]
        neighbors = []
        for dr, dc in directions:
            nr = r + dr
            nc = c + dc
            # check indices
            if 0<=nr<rows and 0<=nc<cols:
                # check if the neighbor has greater or equal value
                if heights[nr][nc] >= heights[r][c]:
                    neighbors.append((nr,nc))

        return neighbors

    def bfs_for_pacific(heights):
        queue = deque([])
        visited_pacific = set()

        for i in range(len(heights)):
            for j in range(len(heights[0])):
                if i == 0 or j == 0:
                    queue.append((i,j))
                    visited_pacific.add((i,j))

        # so now queue has all the elements related attached to the ocean
        # and added all the ocean related nodes to the queue
        while queue:
            n = len(queue)
            for _ in range(n):
                 r,c = queue.popleft()
                 for nr, nc in get_neighbors(heights, r, c):
                     if (nr,nc) not in visited_pacific:
                         queue.append((nr,nc))
                         visited_pacific.add((nr,nc))

        # our visited function contains all the nodes reachable to pacific
        return visited_pacific
    
    def bfs_for_atlantic(heights):
        queue = deque([])
        visited_atlantic = set()

        for i in range(len(heights)-1, -1, -1):
            for j in range(len(heights[0])-1, -1, -1):
                if i == len(heights)-1 or j == len(heights[0])-1:
                    queue.append((i,j))
                    visited_atlantic.add((i,j))

        # so now queue has all the elements related attached to the ocean
        # and added all the ocean related nodes to the queue
        while queue:
            n = len(queue)
            for _ in range(n):
                 r,c = queue.popleft()
                 for nr, nc in get_neighbors(heights, r, c):
                     if (nr,nc) not in visited_atlantic:
                         queue.append((nr,nc))
                         visited_atlantic.add((nr,nc))

        return visited_atlantic

    reachable_from_pacific = bfs_for_pacific(heights)
    reachable_from_atlantic = bfs_for_atlantic(heights)
    # we do this because the sets are unordered and we need ordered list as the output
    reachable_from_both = sorted(
    (list(cell) for cell in reachable_from_pacific & reachable_from_atlantic),
    key=lambda x: (x[0], x[1])
    )

    #alternative for sorting
    # result = [[r, c] for r in range(m) for c in range(n)
    #         if (r, c) in pacific and (r, c) in atlantic]
    return reachable_from_both

### Time and Space Complexity

Mostly right. One fix on time and one on space.

Time

Each BFS is O(mn). Every cell enters the queue at most once because of visited, and each cell checks a constant 4 directions. Correct.
Your sort instinct is right, but “n” is overloaded here, since n is already the column count. The intersection holds k cells, where k can be as large as mn. So the sort costs O(k log k), which in the worst case is O(mn log(mn)).
That sort is the biggest term, so your total time is O(mn log(mn)), not O(mn).

Space: O(mn) is correct, but the reason is off. The guaranteed O(mn) comes from the two visited sets (up to mn cells each) plus the result list. Lead with those. The queue is the weaker argument, since it only ever holds the current frontier plus what’s been added, and it’s hard to claim it hits mn.